# Model initiation

In [ ]:
from faster_whisper import WhisperModel, BatchedInferencePipeline
from torch.cuda import is_available

device = 'cuda' if is_available() else 'cpu'
model_size = 'turbo'
model = WhisperModel(model_size, device=device, compute_type='int8_float16')
model = BatchedInferencePipeline(model)

# Transcribe

In [20]:
from glob import glob
from tqdm import tqdm

vid_paths = glob(r'D:\AIC_LITE\vids\Videos_L01\*.mp4')
transcriptions = {}

for path in tqdm(vid_paths):
    segments, info = model.transcribe(
        path,
        language='vi',
        beam_size=5,
        vad_filter=True,
        batch_size=24,
        word_timestamps=True,
    )
    code = path.split('\\')[-1].split('.')[0]

    trans_words = []
    for segment in segments:
        # print(f'{segment.start:.2f}', f'{segment.end:.2f}', segment.text)
        if not segment.words: continue
        trans_words += [{'start_time': round(float(word.start), 3), 'end_time': round(float(word.end), 3), 'text': word.word.strip()} for word in segment.words]
    transcriptions[code] = trans_words

100%|██████████| 31/31 [11:25<00:00, 22.12s/it]


# Saving

In [ ]:
import json

for code, trans_word in tqdm(list(transcriptions.items())):
    parent = r'..\\additional_data\\transcript_vid'
    with open(f'{parent}\\{code}.json', 'w', encoding='utf-8') as f:
        json.dump(trans_word, f)

100%|██████████| 31/31 [00:00<00:00, 87.16it/s]
